In [7]:
from pymongo import MongoClient
import os
import javalang
from typing import List
import shutil
# Load environment
from dotenv import load_dotenv
# OpenAPI model
import openai
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import Language
from langchain_openai import OpenAIEmbeddings  
# Embedding model
from langchain.document_loaders import TextLoader
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

In [8]:
def connect_mongodb(uri, db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def get_third_parties_with_stop_share(collection):
    # Query to find all documents where "stop_share" is "yes"
    results = collection.find({"stop_share": "yes"}, {"third_party": 1, "_id": 0})
    third_parties = [item["third_party"] for item in results]
    return third_parties
def get_files_with_extension(directory, extension, prefix=""):
    """
    Get all files in the given directory that start with a specific prefix
    and have a specific file extension.

    Args:
        directory (str): Path to the directory.
        extension (str): File extension (e.g., '.java').
        prefix (str): Optional prefix the filename must start with (e.g., 'wrap-').

    Returns:
        list: List of full file paths matching the extension and prefix.
    """
    return [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if os.path.isfile(os.path.join(directory, f))
        and f.lower().endswith(extension.lower())
        and f.startswith(prefix)
    ]
def is_file_empty(file_path: str) -> bool:
    """
    Checks whether the given file is empty (i.e., size = 0 bytes).

    Args:
        file_path (str): Path to the file.

    Returns:
        bool: True if the file is empty, False otherwise.
    """
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"❌ File not found: {file_path}")
    
    return os.path.getsize(file_path) == 0

def index_java_code(file_path: str, faiss_index_path: str, index_name: str = "index"):
    # 1. Load Java code from file
    loader = TextLoader(file_path, encoding="utf-8")
    documents = loader.load()

    # ✅ 1.5: Thêm metadata cho mỗi tài liệu gốc
    for doc in documents:
        doc.metadata["source"] = file_path  # lưu tên file vào metadata

    # 2. Chunk Java code using LangChain splitter
    splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.JAVA,
        chunk_size=1000,
        chunk_overlap=100,
    )
    chunks = splitter.split_documents(documents)

    # 🛑 Skip if no chunks (e.g., empty or unprocessable file)
    if len(chunks) == 0:
        print(f"⚠️ Skipped (no content): {file_path}")
        return

    print(f"📄 File: {file_path} | Chunks created: {len(chunks)}")

    # 3. Create embedding model
    embeddings = OpenAIEmbeddings()

    # 4. Load existing FAISS index or create new
    faiss_file = os.path.join(faiss_index_path, f"{index_name}.faiss")
    if os.path.exists(faiss_file):
        vectorstore = FAISS.load_local(
            folder_path=faiss_index_path,
            index_name=index_name,
            embeddings=embeddings,
            allow_dangerous_deserialization=True
        )
        vectorstore.add_documents(chunks)
        print(f"✅ Added to existing FAISS index.")
    else:
        vectorstore = FAISS.from_documents(chunks, embeddings)
        print(f"🆕 Created new FAISS index.")

    # 5. Save the updated FAISS index
    vectorstore.save_local(faiss_index_path, index_name=index_name)

    # 6. Optional: show total documents (for debug)
    print(f"📦 Total documents in index: {len(vectorstore.docstore._dict)}\n")

In [9]:
# server
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'third_party_services'
# code direction
root_directory = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-rag-for-sdk\rag-external-data"
# vectorDB direction
vector_db_path = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-rag-for-sdk\vectorDB_from_JAVA"

In [10]:
# Setup model
# Load environment variables
load_dotenv()

# Retrieve API key
api_key = os.getenv("OPENAI_API_KEY")
# Ensure the API key is correctly set
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables")

# Initialize the ChatOpenAI model
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    openai_api_key=api_key  # Ensure you explicitly pass the API key
)

In [11]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)
third_parties_with_stop_share = get_third_parties_with_stop_share(collection)
print(third_parties_with_stop_share)

['firebase', 'facebook', 'cleverTap', 'appLovin', 'kochava', 'braze', 'sentry', 'appsflyer', 'vungle', 'baidu', 'urban airship', 'branch', 'ironsource', 'ogury', 'moengage', 'Instabug', 'pushwoosh', 'bugsnag', 'onesignal', 'smaato', 'repro', 'batch', 'Taboola', 'Smart AdServer', 'Adjust', 'Samsung Health', 'Kakao', 'Heap Analytics', 'Digital Turbine', 'PubNative', 'Unity Ads', 'Braintree', 'Iterable', 'New Relic', 'Mixpanel', 'MobileFuse', 'chartbeat', 'swrve', 'Adobe', 'Naver', 'Adyen', 'HyprMX', 'Blueshift', 'optimizely', 'Chartboost', 'MParticle', 'UXCam', 'LogRocket', 'ACRA', 'Smartlook', 'Singular', 'Contentsquare', 'Localytics', 'CustomerIO', 'Sailthru']


In [12]:
for i in range(len(third_parties_with_stop_share)):
    print("---------------- Loop-"+str(i)+" ----------------")
    third_party_name = third_parties_with_stop_share[i]
    print("Third party name: ",third_party_name)
    code_directory_path = os.path.join(root_directory, third_party_name)
    print("Directory path: ",code_directory_path)
    array_file = get_files_with_extension(code_directory_path, ".java", prefix="code-")
    print("Array file: ",array_file)
    for j in range(len(array_file)):
        file_path = array_file[j]
        file_size_check = is_file_empty(file_path)
        if(not file_size_check):
            print("File path: ",file_path)
            vectorstore = index_java_code(
                file_path=file_path,
                faiss_index_path=vector_db_path,
                index_name="index" 
            )
            # print(f"📄 Total documents in index: {len(vectorstore.docstore._dict)}")
            # doc_count = len(vectorstore.docstore._dict)
            # print(f"✅ Indexed: {file_path} | Total documents now: {doc_count}")
    # break

---------------- Loop-0 ----------------
Third party name:  firebase
Directory path:  C:\Users\ASUS\anaconda3-project-code\wearable-2-rag-for-sdk\rag-external-data\firebase
Array file:  ['C:\\Users\\ASUS\\anaconda3-project-code\\wearable-2-rag-for-sdk\\rag-external-data\\firebase\\code-rag-0.java']
File path:  C:\Users\ASUS\anaconda3-project-code\wearable-2-rag-for-sdk\rag-external-data\firebase\code-rag-0.java
📄 File: C:\Users\ASUS\anaconda3-project-code\wearable-2-rag-for-sdk\rag-external-data\firebase\code-rag-0.java | Chunks created: 1
🆕 Created new FAISS index.
📦 Total documents in index: 1

---------------- Loop-1 ----------------
Third party name:  facebook
Directory path:  C:\Users\ASUS\anaconda3-project-code\wearable-2-rag-for-sdk\rag-external-data\facebook
Array file:  ['C:\\Users\\ASUS\\anaconda3-project-code\\wearable-2-rag-for-sdk\\rag-external-data\\facebook\\code-rag-0.java', 'C:\\Users\\ASUS\\anaconda3-project-code\\wearable-2-rag-for-sdk\\rag-external-data\\facebook\\c

In [13]:
# Load FAISS index sau khi indexing xong
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.load_local(
    folder_path=vector_db_path,
    index_name="index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

# In số lượng document hiện tại
print(f"📄 Total documents in FAISS index: {len(vectorstore.docstore._dict)}")

📄 Total documents in FAISS index: 91
